In [1]:
# All library importing
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# Load dataset
df = pd.read_csv("Merged_Unemployment_Data.csv")

# Convert date column to datetime
df['REF_DATE'] = pd.to_datetime(df['REF_DATE'], format='%Y')
# REF_DATE as index
df.set_index('REF_DATE', inplace=True)

# Selecting target variable Actual_incidents and relevant features
target = 'Actual_incidents'
features = ['Rate_per_100000_population', 'Cleared_by_charge','Cleared_otherwise','Percentage_contribution_to_the_Crime_Severity_Index_(CSI)','Rate_adult_charged_per_100000_population_aged_18_years_and_over','Rate_total_persons_charged_per_100000_population_aged_12_years_and_over','Rate_youth_charged_per_100000_population_aged_12_to_17_years','Rate_youth_not_charged_per_100000_population_aged_12_to_17_years','Total_cleared','Total_adult_charged','Total_persons_charged','Total_youth_charged','Total_youth_not_charged']

# Train-test split with 1998-2018 in training data and 2019-2023 in testing data
train = df['1998-01-01' : '2018-12-31']
test = df['2019-01-01' : '2023-12-31']


# Scaling features
scaler = StandardScaler()
X_train = scaler.fit_transform(train[features])
X_test = scaler.transform(test[features])
y_train = train[target].values
y_test = test[target].values
print(f"Training dataset : {X_train.shape}, {len(X_train)/len(df)*100:.2f}%")
print(f"Testing dataset : {X_test.shape}, {len(X_test)/len(df)*100:.2f}%")

Training dataset : (224090, 13), 73.10%
Testing dataset : (82460, 13), 26.90%


In [ ]:
# Train Random Forest Model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

In [ ]:
# Count occurrences of each unique violation
violation_counts = df['Violations'].value_counts().reset_index()
violation_counts.columns = ['Violation', 'Count']

# Display the result
print(violation_counts)

In [2]:
# Function to evaluate models and plot results

# # Define your list of violations
# violations = [
#     'Total robbery [160]',
#     'Total property crime violations [200]',
#     'Total theft under $5,000 (non-motor vehicle) [240]',
#     'Total theft of motor vehicle [220]',
#     'Total mischief [250]',
#     'Total drug violations [401]',
#     'Murder, first degree [1110]',
#     'Murder, second degree [1120]'
# ]

# Place = 'Ontario [35]'

# def evaluate_model(model, X_test, y_test, model_name, X_test3, vio):
#     y_pred = model.predict(X_test)
#     y_pred3 = model.predict(X_test3)
    
#     mae = mean_absolute_error(y_test, y_pred)
#     mse = mean_squared_error(y_test, y_pred)
#     rmse = np.sqrt(mse)
#     r2 = r2_score(y_test, y_pred)
#     accuracy = 100 - (mae / np.mean(y_test) * 100)

#     print(f"{model_name} Performance:")
#     print(f"MAE: {mae:.3f}")
#     print(f"MSE: {mse:.3f}")
#     print(f"RMSE: {rmse:.3f}")
#     print(f"R² Score: {r2:.3f}")
#     print(f"Approximate Accuracy: {accuracy:.2f}%\n")

#     # Ensure same index and length
#     y_pred_series = pd.Series(y_pred3, index=y_test3.index)
    
#     plt.figure(figsize=(10, 6))
#     plt.plot(y_train3, label='Training Data')
#     plt.plot(y_test3, label='Actual Test Data')
#     plt.plot(y_pred_series, label='Predicted Test Data', color='red')
#     plt.title(f'{model_name} - Actual vs Predicted Crime Incidents')
#     plt.suptitle(f'{Place} - {vio}')
#     plt.xlabel('Year')
#     plt.ylabel('Actual_incidents')
#     plt.legend()
#     plt.grid(True)
#     plt.show()



# for vio in violations:
#     print(f"\n--- Evaluating: {vio} ---")
    
#     # Filter the data
#     filtered_df = df[(df['GEO'] == Place) & (df['Violations'] == vio)]
#     y_full = filtered_df[['Actual_incidents']]
#     X_full = filtered_df[[
#         'Rate_per_100000_population',
#         'Cleared_by_charge', 'Cleared_otherwise',
#         'Percentage_contribution_to_the_Crime_Severity_Index_(CSI)',
#         'Rate_adult_charged_per_100000_population_aged_18_years_and_over',
#         'Rate_total_persons_charged_per_100000_population_aged_12_years_and_over',
#         'Rate_youth_charged_per_100000_population_aged_12_to_17_years',
#         'Rate_youth_not_charged_per_100000_population_aged_12_to_17_years',
#         'Total_cleared', 'Total_adult_charged', 'Total_persons_charged',
#         'Total_youth_charged', 'Total_youth_not_charged'
#     ]]
    
#     # Split train/test
#     y_train3 = y_full['1998-01-01' : '2019-12-31']
#     y_test3 = y_full['2019-01-01' : '2023-12-31']
#     X_train = X_full['1998-01-01' : '2019-12-31']
#     X_test = X_full['2019-01-01' : '2023-12-31']

#     # Scale data
#     scaler = StandardScaler()
#     X_train_scaled = scaler.fit_transform(X_train)
#     X_test_scaled = scaler.transform(X_test)

#     # Train model for each violation
#     rf_model = RandomForestRegressor()
#     rf_model.fit(X_train_scaled, y_train3.values.ravel())

#     # Evaluate
#     evaluate_model(rf_model, X_test_scaled, y_test3, f"Random Forest ({vio})", X_test_scaled, vio)

#     # === Additional Evaluations only for Drug Violations ===
#     if vio == 'Total drug violations [401]':
#         # Case 1: Train till 2016, Test 2017–2023
#         y_train3_extra = y_full['1998-01-01':'2016-12-31']
#         y_test3_extra = y_full['2017-01-01':'2023-12-31']
#         X_train_extra = X_full['1998-01-01':'2016-12-31']
#         X_test_extra = X_full['2017-01-01':'2023-12-31']

#         scaler_extra = StandardScaler()
#         X_train_extra_scaled = scaler_extra.fit_transform(X_train_extra)
#         X_test_extra_scaled = scaler_extra.transform(X_test_extra)

#         rf_model_extra = RandomForestRegressor()
#         rf_model_extra.fit(X_train_extra_scaled, y_train3_extra.values.ravel())

#         evaluate_model(rf_model_extra, X_test_extra_scaled, y_test3_extra,
#                        "Random Forest (Drug - Train till 2016, Test till 2023)",
#                        X_test_extra_scaled, y_train3_extra, y_test3_extra, vio)

#         # Case 2: Train till 2016, Test 2017–2018
#         y_test3_short = y_full['2017-01-01':'2018-12-31']
#         X_test_short = X_full['2017-01-01':'2018-12-31']
#         X_test_short_scaled = scaler_extra.transform(X_test_short)

#         evaluate_model(rf_model_extra, X_test_short_scaled, y_test3_short,
#                        "Random Forest (Drug - Train till 2016, Test till 2018)",
#                        X_test_short_scaled, y_train3_extra, y_test3_short, vio)




        

# Try other violations too, and analysis of accuracy between them - done | Also Tried to enhance Accuracy - done
# How deep learning help in increase the accuracy - done
# How many reserach paper is there related to this project - find out but need to read yet



# for drug violation - train till 2016 and then make target till end, also till 2018, and if get better accuracy then we can mention that - done
# consider other provinces too - BC, AB, overall canada (afterwards) - done
# Consider and connect external factors - unemployment (added but not improvement in accuracy), immigration percentage (not getting compatible dataset)
# Consider the model too - decision tree, SVM (don't use), Deep learning - used many, but decision tree is good - as pf now 3 models are good
# Research paper write up planning - models, experiments, about data gathering and processing - done


# Improve Data gathering and processing (See visualization of data analysis) - put one graph of whole crime for all provinces
# Add violation of child/women crime/trafficking - confustion: Sexual interference [1345] (for age below 16), Sexual assault, level 1 [1330] (for crime against women) - but both are sexual, so Abduction under age 14, by parent or guardian [1560], Criminal harassment [1625] - done 

# Data source of unemployment data - done
# Relationship between crime and unemployment data - done
# How many records available for each violations,  and simple table - too much rows, don't understand how to put it


# Mention type of crime - done
# Add Education rate and check how it impact - in dataset what we want is only available for 2012 then dataset is archieved, so not updated
# Correlation of features between target - done
# Make a final code file.. So change can reflect at whole code - done


# Why lightgbm is straight line - some parameters needed and it fixed with better accuracy
# Instead take all the features, take only top 5 features, top 10 features and analysis the result, this is called ablation study, and make 3 windows side by side and analyse, also make rough notes on that - done
# Crime type categorization in table form with columns category, violations, number of records of that violations - done

# Go through the website given by prof. And finding questions


# Before 14: 
# Not mention mlp, lightgbm - done  (updatation of graphs remaining)
# Implement Elastic net, and add if good - done
# Put ablation study - done
# And shift to springer - done

# Before 21: 
# Abstarct, Introduction, literature review - all together conclude in 3 pages
# Abstarct, Introduction, literature review, discussion, conclusion, all else sections
# Ablation study - bar diagram instead of table - done
# Abstract - 4 things - take reference from shared paper - done
# Literature review - 3 Domains - crime, Canada, regression models
# Reference - 15 to 20


# Murder - remove related to it. - done
# Add sexual assault crime Sexual assault, level 3, aggravated [1310] - done
# Section 5.4: Remove population rate, somewhere - done
# In abstract do only for Ontario not for Canadian provinces - done
# Choose only one from MAE, RMSE, R2 - done (Kept only R2) 
# Sexual Assault in ablation study
# Ablation study result table

# Version of libraries put them in GitHub readme - done

# Matplotlib save fig pdf - done

# 65 line - 306,551 (verify it) - done

# While criminological theory - any evidence? Replace: there can be any link between unemployment and crime - done

# Don't put references everywhere, remove from conclusion - done

# Remove using a large national dataset - done

# Line 643 change accurate word - a better forecasting can help... - done

# See any accurate, perfect words.. - done

# 637 cannabis legality reference - give footnotes - 3 out of 5 - done 

# Give the description of ablation figure - done

# Remove correlation analysis from preprocessing - done

# Correlation analysis appeared before ablation study so maintain that - put section 5.5 inside ablation study - done

# All fig and table should mentioned in write up - done

# Check model names.. not capital first character everywhere - done

# 326 classification word replacement - done

# 276 remove government word - done

# Unemployment data - 1. result, diagram - done
# 1. Do everything with unemployment data
# 2. Do both and comparison 

# Line 255: just remove for most violations, use only 2 metrics - done

# 247 mention about extra trees straight line, mention python and libraries versions - done

# Correct model number and forget to mention elastic net - done

# Remove formulas from models - done

# 124 remove indexed by Ontario - done



# Possible integration of Introduction (1) and Literature Review (2) - done
# Section 3.3 Canada’s Uniform Crime Reporting (UCR) classification reference - done
# Possible integration of Models (4) and first paragraph of Model Selection Rationale (4.7) - done
# Section 5.3 is confusing - done
# Section 5.4 ...using using Statistics Canada's categories. - done 
# Avoid possible repetition in section 6 and 7.
# Tables titles and Figure captions - done
# GitHub repo

In [ ]:
violations = [
    'Total robbery [160]',
    'Total property crime violations [200]',
    'Total theft under $5,000 (non-motor vehicle) [240]',
    'Total theft of motor vehicle [220]',
    'Total mischief [250]',
    'Total drug violations [401]',
    # 'Murder, first degree [1110]',
    # 'Murder, second degree [1120]',
    "Sexual assault, level 3, aggravated [1310]",
    'Abduction under age 14, by parent or guardian [1560]',
    'Criminal harassment [1625]'
]

Place = 'Ontario [35]'

def evaluate_model(model, X_test, y_test, model_name, X_test3, y_train3, y_test3, vio):
    y_pred = model.predict(X_test)
    y_pred3 = model.predict(X_test3)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    accuracy = 100 - (mae / np.mean(y_test) * 100)

    print(f"{model_name} Performance:")
    print(f"MAE: {mae:.3f}")
    print(f"MSE: {mse:.3f}")
    print(f"RMSE: {rmse:.3f}")
    print(f"R² Score: {r2:.3f}")
    print(f"Approximate Accuracy: {accuracy:.2f}%\n")

    y_pred_series = pd.Series(y_pred3, index=y_test3.index)

    plt.figure(figsize=(10, 6))
    plt.plot(y_train3, label='Training Data')
    plt.plot(y_test3, label='Actual Test Data')
    plt.plot(y_pred_series, label='Predicted Test Data', color='red')
    plt.title(f'{model_name} - Actual vs Predicted Crime Incidents')
    plt.suptitle(f'{Place} - {vio}')
    plt.xlabel('Year')
    plt.ylabel('Actual_incidents')
    plt.legend()
    plt.grid(True)
    plt.show()


for vio in violations:
    print(f"\n--- Evaluating: {vio} ---")
    
    filtered_df = df[(df['GEO'] == Place) & (df['Violations'] == vio)]
    y_full = filtered_df[['Actual_incidents']]
    X_full = filtered_df[[
        'Rate_per_100000_population',
        'Cleared_by_charge', 'Cleared_otherwise',
        'Percentage_contribution_to_the_Crime_Severity_Index_(CSI)',
        'Rate_adult_charged_per_100000_population_aged_18_years_and_over',
        'Rate_total_persons_charged_per_100000_population_aged_12_years_and_over',
        'Rate_youth_charged_per_100000_population_aged_12_to_17_years',
        'Rate_youth_not_charged_per_100000_population_aged_12_to_17_years',
        'Total_cleared', 'Total_adult_charged', 'Total_persons_charged',
        'Total_youth_charged', 'Total_youth_not_charged'
    ]]

    # === Original Evaluation (Train till 2019, Test till 2023) ===
    y_train3 = y_full['1998-01-01':'2019-12-31']
    y_test3 = y_full['2019-01-01':'2023-12-31']
    X_train = X_full['1998-01-01':'2019-12-31']
    X_test = X_full['2019-01-01':'2023-12-31']

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    rf_model = RandomForestRegressor()
    rf_model.fit(X_train_scaled, y_train3.values.ravel())

    evaluate_model(rf_model, X_test_scaled, y_test3, f"Random Forest ({vio})", X_test_scaled, y_train3, y_test3, vio)

    # === Additional Evaluations only for Drug Violations ===
    if vio == 'Total drug violations [401]':
        # Case 1: Train till 2016, Test 2017–2023
        y_train3_extra = y_full['1998-01-01':'2016-12-31']
        y_test3_extra = y_full['2016-01-01':'2023-12-31']
        X_train_extra = X_full['1998-01-01':'2016-12-31']
        X_test_extra = X_full['2016-01-01':'2023-12-31']

        scaler_extra = StandardScaler()
        X_train_extra_scaled = scaler_extra.fit_transform(X_train_extra)
        X_test_extra_scaled = scaler_extra.transform(X_test_extra)

        rf_model_extra = RandomForestRegressor()
        rf_model_extra.fit(X_train_extra_scaled, y_train3_extra.values.ravel())

        evaluate_model(rf_model_extra, X_test_extra_scaled, y_test3_extra,
                       "Random Forest (Drug - Train till 2016, Test till 2023)",
                       X_test_extra_scaled, y_train3_extra, y_test3_extra, vio)

        # Case 2: Train till 2016, Test 2017–2018
        y_test3_short = y_full['2016-01-01':'2018-12-31']
        X_test_short = X_full['2016-01-01':'2018-12-31']
        X_test_short_scaled = scaler_extra.transform(X_test_short)

        evaluate_model(rf_model_extra, X_test_short_scaled, y_test3_short,
                       "Random Forest (Drug - Train till 2016, Test till 2018)",
                       X_test_short_scaled, y_train3_extra, y_test3_short, vio)

In [ ]:
violations = [
    'Total robbery [160]',
    'Total property crime violations [200]',
    'Total theft under $5,000 (non-motor vehicle) [240]',
    'Total theft of motor vehicle [220]',
    'Total mischief [250]',
    'Total drug violations [401]',
    # 'Murder, first degree [1110]',
    # 'Murder, second degree [1120]',
    "Sexual assault, level 3, aggravated [1310]",
    'Abduction under age 14, by parent or guardian [1560]',
    'Criminal harassment [1625]'
]

Place = 'Alberta [48]'

def evaluate_model(model, X_test, y_test, model_name, X_test3, y_train3, y_test3, vio):
    y_pred = model.predict(X_test)
    y_pred3 = model.predict(X_test3)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    accuracy = 100 - (mae / np.mean(y_test) * 100)

    print(f"{model_name} Performance:")
    print(f"MAE: {mae:.3f}")
    print(f"MSE: {mse:.3f}")
    print(f"RMSE: {rmse:.3f}")
    print(f"R² Score: {r2:.3f}")
    print(f"Approximate Accuracy: {accuracy:.2f}%\n")

    y_pred_series = pd.Series(y_pred3, index=y_test3.index)

    plt.figure(figsize=(10, 6))
    plt.plot(y_train3, label='Training Data')
    plt.plot(y_test3, label='Actual Test Data')
    plt.plot(y_pred_series, label='Predicted Test Data', color='red')
    plt.title(f'{model_name} - Actual vs Predicted Crime Incidents')
    plt.suptitle(f'{Place} - {vio}')
    plt.xlabel('Year')
    plt.ylabel('Actual_incidents')
    plt.legend()
    plt.grid(True)
    plt.show()


for vio in violations:
    print(f"\n--- Evaluating: {vio} ---")
    
    filtered_df = df[(df['GEO'] == Place) & (df['Violations'] == vio)]
    y_full = filtered_df[['Actual_incidents']]
    X_full = filtered_df[[
        'Rate_per_100000_population',
        'Cleared_by_charge', 'Cleared_otherwise',
        'Percentage_contribution_to_the_Crime_Severity_Index_(CSI)',
        'Rate_adult_charged_per_100000_population_aged_18_years_and_over',
        'Rate_total_persons_charged_per_100000_population_aged_12_years_and_over',
        'Rate_youth_charged_per_100000_population_aged_12_to_17_years',
        'Rate_youth_not_charged_per_100000_population_aged_12_to_17_years',
        'Total_cleared', 'Total_adult_charged', 'Total_persons_charged',
        'Total_youth_charged', 'Total_youth_not_charged'
    ]]

    # === Original Evaluation (Train till 2019, Test till 2023) ===
    y_train3 = y_full['1998-01-01':'2019-12-31']
    y_test3 = y_full['2019-01-01':'2023-12-31']
    X_train = X_full['1998-01-01':'2019-12-31']
    X_test = X_full['2019-01-01':'2023-12-31']

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    rf_model = RandomForestRegressor()
    rf_model.fit(X_train_scaled, y_train3.values.ravel())

    evaluate_model(rf_model, X_test_scaled, y_test3, f"Random Forest ({vio})", X_test_scaled, y_train3, y_test3, vio)

    # === Additional Evaluations only for Drug Violations ===
    if vio == 'Total drug violations [401]':
        # Case 1: Train till 2016, Test 2017–2023
        y_train3_extra = y_full['1998-01-01':'2016-12-31']
        y_test3_extra = y_full['2016-01-01':'2023-12-31']
        X_train_extra = X_full['1998-01-01':'2016-12-31']
        X_test_extra = X_full['2016-01-01':'2023-12-31']

        scaler_extra = StandardScaler()
        X_train_extra_scaled = scaler_extra.fit_transform(X_train_extra)
        X_test_extra_scaled = scaler_extra.transform(X_test_extra)

        rf_model_extra = RandomForestRegressor()
        rf_model_extra.fit(X_train_extra_scaled, y_train3_extra.values.ravel())

        evaluate_model(rf_model_extra, X_test_extra_scaled, y_test3_extra,
                       "Random Forest (Drug - Train till 2016, Test till 2023)",
                       X_test_extra_scaled, y_train3_extra, y_test3_extra, vio)

        # Case 2: Train till 2016, Test 2017–2018
        y_test3_short = y_full['2016-01-01':'2018-12-31']
        X_test_short = X_full['2016-01-01':'2018-12-31']
        X_test_short_scaled = scaler_extra.transform(X_test_short)

        evaluate_model(rf_model_extra, X_test_short_scaled, y_test3_short,
                       "Random Forest (Drug - Train till 2016, Test till 2018)",
                       X_test_short_scaled, y_train3_extra, y_test3_short, vio)

In [ ]:
violations = [
    'Total robbery [160]',
    'Total property crime violations [200]',
    'Total theft under $5,000 (non-motor vehicle) [240]',
    'Total theft of motor vehicle [220]',
    'Total mischief [250]',
    'Total drug violations [401]',
    # 'Murder, first degree [1110]',
    # 'Murder, second degree [1120]',
    "Sexual assault, level 3, aggravated [1310]",
    'Abduction under age 14, by parent or guardian [1560]',
    'Criminal harassment [1625]'
]

Place = 'British Columbia [59]'

def evaluate_model(model, X_test, y_test, model_name, X_test3, y_train3, y_test3, vio):
    y_pred = model.predict(X_test)
    y_pred3 = model.predict(X_test3)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    accuracy = 100 - (mae / np.mean(y_test) * 100)

    print(f"{model_name} Performance:")
    print(f"MAE: {mae:.3f}")
    print(f"MSE: {mse:.3f}")
    print(f"RMSE: {rmse:.3f}")
    print(f"R² Score: {r2:.3f}")
    print(f"Approximate Accuracy: {accuracy:.2f}%\n")

    y_pred_series = pd.Series(y_pred3, index=y_test3.index)

    plt.figure(figsize=(10, 6))
    plt.plot(y_train3, label='Training Data')
    plt.plot(y_test3, label='Actual Test Data')
    plt.plot(y_pred_series, label='Predicted Test Data', color='red')
    plt.title(f'{model_name} - Actual vs Predicted Crime Incidents')
    plt.suptitle(f'{Place} - {vio}')
    plt.xlabel('Year')
    plt.ylabel('Actual_incidents')
    plt.legend()
    plt.grid(True)
    plt.show()


for vio in violations:
    print(f"\n--- Evaluating: {vio} ---")
    
    filtered_df = df[(df['GEO'] == Place) & (df['Violations'] == vio)]
    y_full = filtered_df[['Actual_incidents']]
    X_full = filtered_df[[
        'Rate_per_100000_population',
        'Cleared_by_charge', 'Cleared_otherwise',
        'Percentage_contribution_to_the_Crime_Severity_Index_(CSI)',
        'Rate_adult_charged_per_100000_population_aged_18_years_and_over',
        'Rate_total_persons_charged_per_100000_population_aged_12_years_and_over',
        'Rate_youth_charged_per_100000_population_aged_12_to_17_years',
        'Rate_youth_not_charged_per_100000_population_aged_12_to_17_years',
        'Total_cleared', 'Total_adult_charged', 'Total_persons_charged',
        'Total_youth_charged', 'Total_youth_not_charged'
    ]]

    # === Original Evaluation (Train till 2019, Test till 2023) ===
    y_train3 = y_full['1998-01-01':'2019-12-31']
    y_test3 = y_full['2019-01-01':'2023-12-31']
    X_train = X_full['1998-01-01':'2019-12-31']
    X_test = X_full['2019-01-01':'2023-12-31']

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    rf_model = RandomForestRegressor()
    rf_model.fit(X_train_scaled, y_train3.values.ravel())

    evaluate_model(rf_model, X_test_scaled, y_test3, f"Random Forest ({vio})", X_test_scaled, y_train3, y_test3, vio)

    # === Additional Evaluations only for Drug Violations ===
    if vio == 'Total drug violations [401]':
        # Case 1: Train till 2016, Test 2017–2023
        y_train3_extra = y_full['1998-01-01':'2016-12-31']
        y_test3_extra = y_full['2016-01-01':'2023-12-31']
        X_train_extra = X_full['1998-01-01':'2016-12-31']
        X_test_extra = X_full['2016-01-01':'2023-12-31']

        scaler_extra = StandardScaler()
        X_train_extra_scaled = scaler_extra.fit_transform(X_train_extra)
        X_test_extra_scaled = scaler_extra.transform(X_test_extra)

        rf_model_extra = RandomForestRegressor()
        rf_model_extra.fit(X_train_extra_scaled, y_train3_extra.values.ravel())

        evaluate_model(rf_model_extra, X_test_extra_scaled, y_test3_extra,
                       "Random Forest (Drug - Train till 2016, Test till 2023)",
                       X_test_extra_scaled, y_train3_extra, y_test3_extra, vio)

        # Case 2: Train till 2016, Test 2017–2018
        y_test3_short = y_full['2016-01-01':'2018-12-31']
        X_test_short = X_full['2016-01-01':'2018-12-31']
        X_test_short_scaled = scaler_extra.transform(X_test_short)

        evaluate_model(rf_model_extra, X_test_short_scaled, y_test3_short,
                       "Random Forest (Drug - Train till 2016, Test till 2018)",
                       X_test_short_scaled, y_train3_extra, y_test3_short, vio)

In [ ]:
violations = [
    'Total robbery [160]',
    'Total property crime violations [200]',
    'Total theft under $5,000 (non-motor vehicle) [240]',
    'Total theft of motor vehicle [220]',
    'Total mischief [250]',
    'Total drug violations [401]',
    # 'Murder, first degree [1110]',
    # 'Murder, second degree [1120]',
    "Sexual assault, level 3, aggravated [1310]",
    'Abduction under age 14, by parent or guardian [1560]',
    'Criminal harassment [1625]'
]

Place = 'Canada'

def evaluate_model(model, X_test, y_test, model_name, X_test3, y_train3, y_test3, vio):
    y_pred = model.predict(X_test)
    y_pred3 = model.predict(X_test3)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    accuracy = 100 - (mae / np.mean(y_test) * 100)

    print(f"{model_name} Performance:")
    print(f"MAE: {mae:.3f}")
    print(f"MSE: {mse:.3f}")
    print(f"RMSE: {rmse:.3f}")
    print(f"R² Score: {r2:.3f}")
    print(f"Approximate Accuracy: {accuracy:.2f}%\n")

    y_pred_series = pd.Series(y_pred3, index=y_test3.index)

    plt.figure(figsize=(10, 6))
    plt.plot(y_train3, label='Training Data')
    plt.plot(y_test3, label='Actual Test Data')
    plt.plot(y_pred_series, label='Predicted Test Data', color='red')
    plt.title(f'{model_name} - Actual vs Predicted Crime Incidents')
    plt.suptitle(f'{Place} - {vio}')
    plt.xlabel('Year')
    plt.ylabel('Actual_incidents')
    plt.legend()
    plt.grid(True)
    plt.show()


for vio in violations:
    print(f"\n--- Evaluating: {vio} ---")
    
    filtered_df = df[(df['GEO'] == Place) & (df['Violations'] == vio)]
    y_full = filtered_df[['Actual_incidents']]
    X_full = filtered_df[[
        'Rate_per_100000_population',
        'Cleared_by_charge', 'Cleared_otherwise',
        'Percentage_contribution_to_the_Crime_Severity_Index_(CSI)',
        'Rate_adult_charged_per_100000_population_aged_18_years_and_over',
        'Rate_total_persons_charged_per_100000_population_aged_12_years_and_over',
        'Rate_youth_charged_per_100000_population_aged_12_to_17_years',
        'Rate_youth_not_charged_per_100000_population_aged_12_to_17_years',
        'Total_cleared', 'Total_adult_charged', 'Total_persons_charged',
        'Total_youth_charged', 'Total_youth_not_charged'
    ]]

    # === Original Evaluation (Train till 2019, Test till 2023) ===
    y_train3 = y_full['1998-01-01':'2019-12-31']
    y_test3 = y_full['2019-01-01':'2023-12-31']
    X_train = X_full['1998-01-01':'2019-12-31']
    X_test = X_full['2019-01-01':'2023-12-31']

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    rf_model = RandomForestRegressor()
    rf_model.fit(X_train_scaled, y_train3.values.ravel())

    evaluate_model(rf_model, X_test_scaled, y_test3, f"Random Forest ({vio})", X_test_scaled, y_train3, y_test3, vio)

    # === Additional Evaluations only for Drug Violations ===
    if vio == 'Total drug violations [401]':
        # Case 1: Train till 2016, Test 2017–2023
        y_train3_extra = y_full['1998-01-01':'2016-12-31']
        y_test3_extra = y_full['2016-01-01':'2023-12-31']
        X_train_extra = X_full['1998-01-01':'2016-12-31']
        X_test_extra = X_full['2016-01-01':'2023-12-31']

        scaler_extra = StandardScaler()
        X_train_extra_scaled = scaler_extra.fit_transform(X_train_extra)
        X_test_extra_scaled = scaler_extra.transform(X_test_extra)

        rf_model_extra = RandomForestRegressor()
        rf_model_extra.fit(X_train_extra_scaled, y_train3_extra.values.ravel())

        evaluate_model(rf_model_extra, X_test_extra_scaled, y_test3_extra,
                       "Random Forest (Drug - Train till 2016, Test till 2023)",
                       X_test_extra_scaled, y_train3_extra, y_test3_extra, vio)

        # Case 2: Train till 2016, Test 2017–2018
        y_test3_short = y_full['2016-01-01':'2018-12-31']
        X_test_short = X_full['2016-01-01':'2018-12-31']
        X_test_short_scaled = scaler_extra.transform(X_test_short)

        evaluate_model(rf_model_extra, X_test_short_scaled, y_test3_short,
                       "Random Forest (Drug - Train till 2016, Test till 2018)",
                       X_test_short_scaled, y_train3_extra, y_test3_short, vio)




print("\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n Model trained with Unemployment data ")

for vio in violations:
    print(f"\n--- Evaluating: {vio} ---")
    
    filtered_df = df[(df['GEO'] == Place) & (df['Violations'] == vio)]
    y_full = filtered_df[['Actual_incidents']]
    X_full = filtered_df[[
        'Rate_per_100000_population',
        'Cleared_by_charge', 'Cleared_otherwise',
        'Percentage_contribution_to_the_Crime_Severity_Index_(CSI)',
        'Rate_adult_charged_per_100000_population_aged_18_years_and_over',
        'Rate_total_persons_charged_per_100000_population_aged_12_years_and_over',
        'Rate_youth_charged_per_100000_population_aged_12_to_17_years',
        'Rate_youth_not_charged_per_100000_population_aged_12_to_17_years',
        'Total_cleared', 'Total_adult_charged', 'Total_persons_charged',
        'Total_youth_charged', 'Total_youth_not_charged', 'Unemployment_rate'
    ]]

    # === Original Evaluation (Train till 2019, Test till 2023) ===
    y_train3 = y_full['1998':'2019']
    y_test3 = y_full['2019':'2023']
    X_train = X_full['1998':'2019']
    X_test = X_full['2019':'2023']

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    rf_model = RandomForestRegressor()
    rf_model.fit(X_train_scaled, y_train3.values.ravel())

    evaluate_model(rf_model, X_test_scaled, y_test3, f"Random Forest ({vio})", X_test_scaled, y_train3, y_test3, vio)

    # === Additional Evaluations only for Drug Violations ===
    if vio == 'Total drug violations [401]':
        # Case 1: Train till 2016, Test 2017–2023
        y_train3_extra = y_full['1998-01-01':'2016-12-31']
        y_test3_extra = y_full['2016-01-01':'2023-12-31']
        X_train_extra = X_full['1998-01-01':'2016-12-31']
        X_test_extra = X_full['2016-01-01':'2023-12-31']

        scaler_extra = StandardScaler()
        X_train_extra_scaled = scaler_extra.fit_transform(X_train_extra)
        X_test_extra_scaled = scaler_extra.transform(X_test_extra)

        rf_model_extra = RandomForestRegressor()
        rf_model_extra.fit(X_train_extra_scaled, y_train3_extra.values.ravel())

        evaluate_model(rf_model_extra, X_test_extra_scaled, y_test3_extra,
                       "Random Forest (Drug - Train till 2016, Test till 2023)",
                       X_test_extra_scaled, y_train3_extra, y_test3_extra, vio)

        # Case 2: Train till 2016, Test 2017–2018
        y_test3_short = y_full['2016-01-01':'2018-12-31']
        X_test_short = X_full['2016-01-01':'2018-12-31']
        X_test_short_scaled = scaler_extra.transform(X_test_short)

        evaluate_model(rf_model_extra, X_test_short_scaled, y_test3_short,
                       "Random Forest (Drug - Train till 2016, Test till 2018)",
                       X_test_short_scaled, y_train3_extra, y_test3_short, vio)

In [ ]:
# merged unemployment data with old prepreocessed data

# # Load dataset
# unemployment_df = pd.read_csv("Unemployment_Data.csv")

# # --- Step 1: Clean up unemployment dataset ---

# # Keep only necessary columns
# unemployment_clean = unemployment_df[
#     ['REF_DATE', 'GEO', 'Labour force characteristics', 'VALUE']
# ]

# # Filter only 'Unemployment rate'
# unemployment_clean = unemployment_clean[
#     unemployment_clean['Labour force characteristics'] == 'Unemployment rate'
# ]

# # Extract year
# unemployment_clean['Year'] = unemployment_clean['REF_DATE'].str.slice(0, 4).astype(int)

# # Standardize province names to match crime dataset (e.g., "Ontario" → "Ontario [35]")
# geo_mapping = {
#     'Ontario': 'Ontario [35]',
#     'Quebec': 'Quebec [24]',
#     'British Columbia': 'British Columbia [59]',
#     'Alberta': 'Alberta [48]',
#     'Manitoba': 'Manitoba [46]',
#     'Saskatchewan': 'Saskatchewan [47]',
#     'Nova Scotia': 'Nova Scotia [12]',
#     'New Brunswick': 'New Brunswick [13]',
#     'Newfoundland and Labrador': 'Newfoundland and Labrador [10]',
#     'Prince Edward Island': 'Prince Edward Island [11]',
#     'Yukon': 'Yukon [60]',
#     'Northwest Territories': 'Northwest Territories [61]',
#     'Nunavut': 'Nunavut [62]'
# }
# unemployment_clean['GEO'] = unemployment_clean['GEO'].replace(geo_mapping)

# # --- Step 2: Compute yearly average unemployment rate per province ---
# unemployment_yearly = unemployment_clean.groupby(['Year', 'GEO'])['VALUE'].mean().reset_index()
# unemployment_yearly.rename(columns={'GEO': 'Province', 'VALUE': 'Unemployment_rate'}, inplace=True)

# # --- Step 3: Prepare crime data for merge ---
# df['Year'] = df['REF_DATE'].str.slice(0, 4).astype(int)  # Already yearly
# df.rename(columns={'GEO': 'Province'}, inplace=True)

# # --- Step 4: Merge both datasets ---
# merged_df = pd.merge(df, unemployment_yearly, how='left', on=['Year', 'Province'])
# print(merged_df)
# merged_df.to_csv('Merged_Crime_Unemployment2.csv', index=False)


In [ ]:
# Define models
# !pip install lightgbm
# All library imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

Place = 'Ontario [35]'
models = {
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    # "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42),
    # "XGBoost": XGBRegressor(n_estimators=100, random_state=42),
    # "LightGBM": LGBMRegressor(n_estimators=100, random_state=42),
    # "MLP Regressor": MLPRegressor(hidden_layer_sizes=(100, 100), max_iter=1000, random_state=42),
    # "SVR": SVR(kernel='rbf', C=100, epsilon=0.1)
}

# Evaluation function
def evaluate_model(model, X_test, y_test, model_name, X_test3, y_train3, y_test3, vio):
    y_pred = model.predict(X_test)
    y_pred3 = model.predict(X_test3)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    accuracy = 100 - (mae / np.mean(y_test) * 100)

    print(f"{model_name} Performance:")
    print(f"MAE: {mae:.3f}")
    print(f"MSE: {mse:.3f}")
    print(f"RMSE: {rmse:.3f}")
    print(f"R² Score: {r2:.3f}")
    print(f"Approximate Accuracy: {accuracy:.2f}%\n")

    y_pred_series = pd.Series(y_pred3, index=y_test3.index)

    plt.figure(figsize=(10, 6))
    plt.plot(y_train3, label='Training Data')
    plt.plot(y_test3, label='Actual Test Data')
    plt.plot(y_pred_series, label='Predicted Test Data', color='red')
    plt.title(f'{model_name} - Actual vs Predicted Crime Incidents')
    plt.suptitle(f'{Place} - {vio}')
    plt.xlabel('Year')
    plt.ylabel('Actual_incidents')
    plt.legend()
    plt.grid(True)
    plt.show()

# Loop through each violation and evaluate all models
for vio in violations:
    print(f"\n--- Evaluating: {vio} ---")
    
    filtered_df = df[(df['GEO'] == Place) & (df['Violations'] == vio)]
    y_full = filtered_df[['Actual_incidents']]
    X_full = filtered_df[features]

    # Split train/test
    y_train3 = y_full['1998-01-01':'2019-12-31']
    y_test3 = y_full['2019-01-01':'2023-12-31']
    X_train = X_full['1998-01-01':'2019-12-31']
    X_test = X_full['2019-01-01':'2023-12-31']

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Train and evaluate each model
    for model_name, model_instance in models.items():
        model_instance.fit(X_train_scaled, y_train3.values.ravel())
        evaluate_model(model_instance, X_test_scaled, y_test3, f"{model_name} ({vio})", X_test_scaled, y_train3, y_test3, vio)

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor


violations = [
    'Total robbery [160]',
    'Total property crime violations [200]',
    'Total theft under $5,000 (non-motor vehicle) [240]',
    'Total theft of motor vehicle [220]',
    'Total mischief [250]',
    'Total drug violations [401]',
    # 'Murder, first degree [1110]',
    # 'Murder, second degree [1120]',
    "Sexual assault, level 3, aggravated [1310]",
    'Abduction under age 14, by parent or guardian [1560]',
    'Criminal harassment [1625]'
]

Place = 'Ontario [35]'
        
def evaluate_model(model, X_test, y_test, model_name, X_test3, y_train3, y_test3, vio):
    y_pred = model.predict(X_test)
    y_pred3 = model.predict(X_test3)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    accuracy = 100 - (mae / np.mean(y_test) * 100)

    print(f"{model_name} Performance:")
    print(f"MAE: {mae:.3f}")
    print(f"MSE: {mse:.3f}")
    print(f"RMSE: {rmse:.3f}")
    print(f"R² Score: {r2:.3f}")
    print(f"Approximate Accuracy: {accuracy:.2f}%\n")

    y_pred_series = pd.Series(y_pred3, index=y_test3.index)

    plt.figure(figsize=(10, 6))
    plt.plot(y_train3, label='Training Data')
    plt.plot(y_test3, label='Actual Test Data')
    plt.plot(y_pred_series, label='Predicted Test Data', color='red')
    plt.title(f'{model_name} - Actual vs Predicted Crime Incidents')
    plt.suptitle(f'{Place} - {vio}')
    plt.xlabel('Year')
    plt.ylabel('Actual_incidents')
    plt.legend()
    plt.grid(True)
    plt.show()


for vio in violations:
    print(f"\n--- Evaluating: {vio} ---")
    
    filtered_df = df[(df['GEO'] == Place) & (df['Violations'] == vio)]
    y_full = filtered_df[['Actual_incidents']]
    X_full = filtered_df[[
        'Rate_per_100000_population',
        'Cleared_by_charge', 'Cleared_otherwise',
        'Percentage_contribution_to_the_Crime_Severity_Index_(CSI)',
        'Rate_adult_charged_per_100000_population_aged_18_years_and_over',
        'Rate_total_persons_charged_per_100000_population_aged_12_years_and_over',
        'Rate_youth_charged_per_100000_population_aged_12_to_17_years',
        'Rate_youth_not_charged_per_100000_population_aged_12_to_17_years',
        'Total_cleared', 'Total_adult_charged', 'Total_persons_charged',
        'Total_youth_charged', 'Total_youth_not_charged'
    ]]

    # === Original Evaluation (Train till 2019, Test till 2023) ===
    y_train3 = y_full['1998-01-01':'2019-12-31']
    y_test3 = y_full['2019-01-01':'2023-12-31']
    X_train = X_full['1998-01-01':'2019-12-31']
    X_test = X_full['2019-01-01':'2023-12-31']

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Random Forest Model
    rf_model = RandomForestRegressor()
    rf_model.fit(X_train_scaled, y_train3.values.ravel())

    
    # Decision Tree Model
    dt_model = DecisionTreeRegressor(random_state=42)
    dt_model.fit(X_train_scaled, y_train3.values.ravel())
    
    # MLP Neural Network
    mlp_model = MLPRegressor(hidden_layer_sizes=(100, 50), activation='relu', solver='adam', max_iter=1000, random_state=42)
    mlp_model.fit(X_train_scaled, y_train3.values.ravel())

    evaluate_model(rf_model, X_test_scaled, y_test3, f"Random Forest ({vio})", X_test_scaled, y_train3, y_test3, vio)
    evaluate_model(dt_model, X_test_scaled, y_test3, f"Decision Tree ({vio})", X_test_scaled, y_train3, y_test3, vio)
    evaluate_model(mlp_model, X_test_scaled, y_test3, f"Neural Net (MLP) ({vio})", X_test_scaled, y_train3, y_test3, vio)

    # === Additional Evaluations only for Drug Violations ===
    if vio == 'Total drug violations [401]':
        # Case 1: Train till 2016, Test till 2023
        y_train3_extra = y_full['1998-01-01':'2016-12-31']
        y_test3_extra = y_full['2016-01-01':'2023-12-31']
        X_train_extra = X_full['1998-01-01':'2016-12-31']
        X_test_extra = X_full['2016-01-01':'2023-12-31']
    
        scaler_extra = StandardScaler()
        X_train_extra_scaled = scaler_extra.fit_transform(X_train_extra)
        X_test_extra_scaled = scaler_extra.transform(X_test_extra)
    
        # Random Forest
        rf_model_extra = RandomForestRegressor()
        rf_model_extra.fit(X_train_extra_scaled, y_train3_extra.values.ravel())
        evaluate_model(rf_model_extra, X_test_extra_scaled, y_test3_extra,
                       "Random Forest (Drug - Train till 2016, Test till 2023)",
                       X_test_extra_scaled, y_train3_extra, y_test3_extra, vio)
    
        # Decision Tree
        dt_model_extra = DecisionTreeRegressor(random_state=42)
        dt_model_extra.fit(X_train_extra_scaled, y_train3_extra.values.ravel())
        evaluate_model(dt_model_extra, X_test_extra_scaled, y_test3_extra,
                       "Decision Tree (Drug - Train till 2016, Test till 2023)",
                       X_test_extra_scaled, y_train3_extra, y_test3_extra, vio)
    
        # MLP Neural Network
        mlp_model_extra = MLPRegressor(hidden_layer_sizes=(100, 50), activation='relu',
                                       solver='adam', max_iter=1000, random_state=42)
        mlp_model_extra.fit(X_train_extra_scaled, y_train3_extra.values.ravel())
        evaluate_model(mlp_model_extra, X_test_extra_scaled, y_test3_extra,
                       "Neural Net (Drug - Train till 2016, Test till 2023)",
                       X_test_extra_scaled, y_train3_extra, y_test3_extra, vio)
    
        # Case 2: Train till 2016, Test 2017–2018
        y_test3_short = y_full['2016-01-01':'2018-12-31']
        X_test_short = X_full['2016-01-01':'2018-12-31']
        X_test_short_scaled = scaler_extra.transform(X_test_short)
    
        evaluate_model(rf_model_extra, X_test_short_scaled, y_test3_short,
                       "Random Forest (Drug - Train till 2016, Test till 2018)",
                       X_test_short_scaled, y_train3_extra, y_test3_short, vio)
    
        evaluate_model(dt_model_extra, X_test_short_scaled, y_test3_short,
                       "Decision Tree (Drug - Train till 2016, Test till 2018)",
                       X_test_short_scaled, y_train3_extra, y_test3_short, vio)
    
        evaluate_model(mlp_model_extra, X_test_short_scaled, y_test3_short,
                       "Neural Net (Drug - Train till 2016, Test till 2018)",
                       X_test_short_scaled, y_train3_extra, y_test3_short, vio)


In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor


violations = [
    'Total robbery [160]',
    'Total property crime violations [200]',
    'Total theft under $5,000 (non-motor vehicle) [240]',
    'Total theft of motor vehicle [220]',
    'Total mischief [250]',
    'Total drug violations [401]',
    # 'Murder, first degree [1110]',
    # 'Murder, second degree [1120]',
    "Sexual assault, level 3, aggravated [1310]",
    'Abduction under age 14, by parent or guardian [1560]',
    'Criminal harassment [1625]'
]

Place = 'Ontario [35]'


def plot_combined_predictions(X_train_scaled, y_train, X_test_scaled, y_test, title_suffix, place_name):
    models = {
        "Random Forest": RandomForestRegressor(),
        "Decision Tree": DecisionTreeRegressor(random_state=42),
        "Neural Net (MLP)": MLPRegressor(hidden_layer_sizes=(100, 50), activation='relu', solver='adam', max_iter=1000, random_state=42),
    }

    predictions = {}

    for name, model in models.items():
        model.fit(X_train_scaled, y_train.values.ravel())
        y_pred = model.predict(X_test_scaled)
        predictions[name] = pd.Series(y_pred, index=y_test.index)

        # Print metrics
        mae = mean_absolute_error(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_test, y_pred)
        accuracy = 100 - (mae / np.mean(y_test) * 100)

        print(f"{name} ({title_suffix}):")
        print(f"  MAE={mae:.2f}, MSE={mse:.2f}, RMSE={rmse:.2f}, R²={r2:.3f}, Accuracy={accuracy:.2f}%")

    # Plotting
    plt.figure(figsize=(12, 6))
    plt.plot(y_train, label='Training Data')
    plt.plot(y_test, label='Actual Test Data', linewidth=2, color='black')

    for name, y_pred_series in predictions.items():
        plt.plot(y_pred_series, label=f'Predicted: {name}')

    plt.title(f'Combined Model Predictions - {title_suffix}')
    plt.suptitle(place_name)
    plt.xlabel('Year')
    plt.ylabel('Actual_incidents')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


for vio in violations:
    print(f"\n--- Evaluating: {vio} ---")
    
    filtered_df = df[(df['GEO'] == Place) & (df['Violations'] == vio)]
    y_full = filtered_df[['Actual_incidents']]
    X_full = filtered_df[[
        'Rate_per_100000_population',
        'Cleared_by_charge', 'Cleared_otherwise',
        'Percentage_contribution_to_the_Crime_Severity_Index_(CSI)',
        'Rate_adult_charged_per_100000_population_aged_18_years_and_over',
        'Rate_total_persons_charged_per_100000_population_aged_12_years_and_over',
        'Rate_youth_charged_per_100000_population_aged_12_to_17_years',
        'Rate_youth_not_charged_per_100000_population_aged_12_to_17_years',
        'Total_cleared', 'Total_adult_charged', 'Total_persons_charged',
        'Total_youth_charged', 'Total_youth_not_charged'
    ]]

    # ========== General Evaluation ==========
    y_train3 = y_full['1998-01-01':'2019-12-31']
    y_test3 = y_full['2019-01-01':'2023-12-31']
    X_train = X_full['1998-01-01':'2019-12-31']
    X_test = X_full['2019-01-01':'2023-12-31']

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    plot_combined_predictions(X_train_scaled, y_train3, X_test_scaled, y_test3, f"{vio} (Train till 2019, Test 2020–2023)", Place)

    # ========== Additional Cases for Drug Violations ==========
    if vio == 'Total drug violations [401]':
        # --- Case 1: Train till 2016, Test 2016–2023 ---
        y_train_1 = y_full['1998-01-01':'2016-12-31']
        y_test_1 = y_full['2016-01-01':'2023-12-31']
        X_train_1 = X_full['1998-01-01':'2016-12-31']
        X_test_1 = X_full['2016-01-01':'2023-12-31']

        scaler_1 = StandardScaler()
        X_train_1_scaled = scaler_1.fit_transform(X_train_1)
        X_test_1_scaled = scaler_1.transform(X_test_1)

        plot_combined_predictions(X_train_1_scaled, y_train_1, X_test_1_scaled, y_test_1,
                                  f"{vio} (Train till 2016, Test 2016–2023)", Place)

        # --- Case 2: Train till 2016, Test 2016–2018 ---
        y_test_2 = y_full['2016-01-01':'2018-12-31']
        X_test_2 = X_full['2016-01-01':'2018-12-31']
        X_test_2_scaled = scaler_1.transform(X_test_2)  # use same scaler

        plot_combined_predictions(X_train_1_scaled, y_train_1, X_test_2_scaled, y_test_2,
                                  f"{vio} (Train till 2016, Test 2016–2018)", Place)


In [ ]:
# !pip install lightgbm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor


violations = [
    'Total robbery [160]',
    'Total property crime violations [200]',
    'Total theft under $5,000 (non-motor vehicle) [240]',
    'Total theft of motor vehicle [220]',
    'Total mischief [250]',
    'Total drug violations [401]',
    'Murder, first degree [1110]',
    'Murder, second degree [1120]',
    'Abduction under age 14, by parent or guardian [1560]',
    'Criminal harassment [1625]'
]

Place = 'Ontario [35]'


def plot_combined_predictions(X_train_scaled, y_train, X_test_scaled, y_test, title_suffix, place_name):
    models = {
        "Random Forest": RandomForestRegressor(),
        "Decision Tree": DecisionTreeRegressor(random_state=42),
        "Neural Net (MLP)": MLPRegressor(hidden_layer_sizes=(100, 50), activation='relu', solver='adam', max_iter=1000, random_state=42),
        "XGBoost": XGBRegressor(),
        "LightGBM": LGBMRegressor(),
        "Ridge Regression": Ridge(),
        "Extra Trees": ExtraTreesRegressor()
    }

    predictions = {}

    for name, model in models.items():
        model.fit(X_train_scaled, y_train.values.ravel())
        y_pred = model.predict(X_test_scaled)
        predictions[name] = pd.Series(y_pred, index=y_test.index)

        # Print metrics
        mae = mean_absolute_error(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_test, y_pred)
        accuracy = 100 - (mae / np.mean(y_test) * 100)

        print(f"{name} ({title_suffix}):")
        print(f"  MAE={mae:.2f}, MSE={mse:.2f}, RMSE={rmse:.2f}, R²={r2:.3f}, Accuracy={accuracy:.2f}%")

    # Plotting
    plt.figure(figsize=(12, 6))
    plt.plot(y_train, label='Training Data')
    plt.plot(y_test, label='Actual Test Data', linewidth=2, color='black')

    for name, y_pred_series in predictions.items():
        plt.plot(y_pred_series, label=f'Predicted: {name}')

    plt.title(f'Combined Model Predictions - {title_suffix}')
    plt.suptitle(place_name)
    plt.xlabel('Year')
    plt.ylabel('Actual_incidents')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


for vio in violations:
    print(f"\n--- Evaluating: {vio} ---")
    
    filtered_df = df[(df['GEO'] == Place) & (df['Violations'] == vio)]
    y_full = filtered_df[['Actual_incidents']]
    X_full = filtered_df[[
        'Rate_per_100000_population',
        'Cleared_by_charge', 'Cleared_otherwise',
        'Percentage_contribution_to_the_Crime_Severity_Index_(CSI)',
        'Rate_adult_charged_per_100000_population_aged_18_years_and_over',
        'Rate_total_persons_charged_per_100000_population_aged_12_years_and_over',
        'Rate_youth_charged_per_100000_population_aged_12_to_17_years',
        'Rate_youth_not_charged_per_100000_population_aged_12_to_17_years',
        'Total_cleared', 'Total_adult_charged', 'Total_persons_charged',
        'Total_youth_charged', 'Total_youth_not_charged'
    ]]

    # ========== General Evaluation ==========
    y_train3 = y_full['1998-01-01':'2019-12-31']
    y_test3 = y_full['2019-01-01':'2023-12-31']
    X_train = X_full['1998-01-01':'2019-12-31']
    X_test = X_full['2019-01-01':'2023-12-31']

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    plot_combined_predictions(X_train_scaled, y_train3, X_test_scaled, y_test3, f"{vio} (Train till 2019, Test 2020–2023)", Place)

    # ========== Additional Cases for Drug Violations ==========
    if vio == 'Total drug violations [401]':
        # --- Case 1: Train till 2016, Test 2016–2023 ---
        y_train_1 = y_full['1998-01-01':'2016-12-31']
        y_test_1 = y_full['2016-01-01':'2023-12-31']
        X_train_1 = X_full['1998-01-01':'2016-12-31']
        X_test_1 = X_full['2016-01-01':'2023-12-31']

        scaler_1 = StandardScaler()
        X_train_1_scaled = scaler_1.fit_transform(X_train_1)
        X_test_1_scaled = scaler_1.transform(X_test_1)

        plot_combined_predictions(X_train_1_scaled, y_train_1, X_test_1_scaled, y_test_1,
                                  f"{vio} (Train till 2016, Test 2016–2023)", Place)

        # --- Case 2: Train till 2016, Test 2016–2018 ---
        y_test_2 = y_full['2016-01-01':'2018-12-31']
        X_test_2 = X_full['2016-01-01':'2018-12-31']
        X_test_2_scaled = scaler_1.transform(X_test_2)  # use same scaler

        plot_combined_predictions(X_train_1_scaled, y_train_1, X_test_2_scaled, y_test_2,
                                  f"{vio} (Train till 2016, Test 2016–2018)", Place)


In [ ]:
# General correlation analysis (all violations) for a specific place
import seaborn as sns

# Filter by place only, include all violations
place_df = df[df['GEO'] == Place]

# Select features + target
features_and_target = [
    'Rate_per_100000_population',
    'Cleared_by_charge', 'Cleared_otherwise',
    'Percentage_contribution_to_the_Crime_Severity_Index_(CSI)',
    'Rate_adult_charged_per_100000_population_aged_18_years_and_over',
    'Rate_total_persons_charged_per_100000_population_aged_12_years_and_over',
    'Rate_youth_charged_per_100000_population_aged_12_to_17_years',
    'Rate_youth_not_charged_per_100000_population_aged_12_to_17_years',
    'Total_cleared', 'Total_adult_charged', 'Total_persons_charged',
    'Total_youth_charged', 'Total_youth_not_charged', 'Actual_incidents'
]

# Filter columns and drop any rows with missing values
corr_df = place_df[features_and_target].dropna()

# Compute correlation matrix
corr_matrix = corr_df.corr(numeric_only=True)

# Plot heatmap of features vs target only
plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix[['Actual_incidents']].sort_values(by='Actual_incidents', ascending=False), 
            annot=True, cmap='coolwarm', cbar=True)
plt.title(f'Feature Correlation with Actual Incidents - {Place}')
plt.tight_layout()
plt.show()


In [ ]:
import seaborn as sns

# Loop through each violation and show correlation heatmap
for vio in violations:
    filtered_df_corr = df[(df['GEO'] == Place) & (df['Violations'] == vio)]

    corr_df = filtered_df_corr[[
        'Rate_per_100000_population',
        'Cleared_by_charge', 'Cleared_otherwise',
        'Percentage_contribution_to_the_Crime_Severity_Index_(CSI)',
        'Rate_adult_charged_per_100000_population_aged_18_years_and_over',
        'Rate_total_persons_charged_per_100000_population_aged_12_years_and_over',
        'Rate_youth_charged_per_100000_population_aged_12_to_17_years',
        'Rate_youth_not_charged_per_100000_population_aged_12_to_17_years',
        'Total_cleared', 'Total_adult_charged', 'Total_persons_charged',
        'Total_youth_charged', 'Total_youth_not_charged', 'Actual_incidents'
    ]]

    correlation_matrix = corr_df.corr(numeric_only=True)

    plt.figure(figsize=(10, 6))
    sns.heatmap(
        correlation_matrix[['Actual_incidents']].sort_values(by='Actual_incidents', ascending=False),
        annot=True, cmap='coolwarm', cbar=True, vmin=-1, vmax=1
    )
    plt.title(f'Feature Correlation with Actual Incidents - {vio}')
    plt.tight_layout()
    plt.show()
